# Linear Regression from Scratch

Linear regression is used when there's a clear linear relationship between a dependent (output) target variable and an independent (input) variable. The result is a line of best fit that tries to get as close to as many points as possible.

---

## The Model

$$ f_{w,b}(x) = w \cdot x + b $$

Where each variable represents:

- $f_{w,b}(x)$: The predicted y value according to the given inputs
- $b$: The y-intercept (the bias)
- $w$: The slope of the line of best fit (the weight)
- $x$: The input value

---

## The Loss / Cost Function

To find the line of best fit we need to calculate the loss, which measures how far the line is from the actual results. The loss function used is Mean Squared Error (MSE), which measures the error by squaring the difference between the predicted and actual value of y. By summing all squared errors and dividing by $2m$ we get the average squared error of the line, telling us how wrong it is on average. The $\frac{1}{2}$ is a convenience factor, it cancels out cleanly when we take the derivative later. Our goal is to minimize this cost.

$$J(w,b) = \frac{1}{2m} \sum_{i=1}^{m} (f_{w,b}(x^{(i)}) - y^{(i)})^2$$

Where:

- $J(w,b)$: The cost
- $m$: The number of training examples
- $i$: Denotes which example we are on
- $x^{(i)}$: The $i$ th input value
- $y^{(i)}$: The $i$ th actual output value

---

## Gradient Descent

To minimize the cost we use gradient descent. We compute the partial derivatives of $J(w,b)$ with respect to $w$ and $b$, these tell us the slope of the cost function at the current values. Since the gradient points uphill, we subtract it from our parameters to move downhill toward the minimum. $\alpha$ is the learning rate, controlling how large each step is.

The update rules are:

$$w := w - \alpha \frac{\partial J(w,b)}{\partial w}$$

$$b := b - \alpha \frac{\partial J(w,b)}{\partial b}$$

Where the partial derivatives expand to:

$$\frac{\partial J(w,b)}{\partial w} = \frac{1}{m} \sum_{i=1}^{m} (f_{w,b}(x^{(i)}) - y^{(i)}) x^{(i)}$$

$$\frac{\partial J(w,b)}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (f_{w,b}(x^{(i)}) - y^{(i)})$$

Notice the derivative for $w$ has an extra $x^{(i)}$ term compared to $b$, which comes from the chain rule, since $w$ is multiplied by $x$ in the model.

---

In [48]:
import math
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import pickle
def standardize(X_train, X_test):

    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std
    
    return X_train, X_test

class LinearRegression:
    
    def __init__(self, learning_rate):
        self.learning_rate = learning_rate
        self.w = np.random.randn() * .01
        self.b = 0
    
    
    def predict(self, x_train):
        return x_train * self.w + self.b

    def loss(self, x_train, y_train):
        m = len(x_train)
        loss = (1/(2*m))* (np.sum((self.predict(x_train) - y_train)**2))
        return loss
    
    def gradient_descent(self, x_train, y_train):
        m = len(x_train)
        weight_partial = (1/m) * np.sum((self.predict(x_train)-y_train) * x_train)
        bias_partial = (1/m) * np.sum(self.predict(x_train)-y_train)
        
        self.w -= self.learning_rate * weight_partial
        self.b -= self.learning_rate * bias_partial
    
    def fit(self, x_train, y_train, i):
        self.x = x_train
        self.y = y_train
        
        losses = []
        
        for i in range(i):
            loss = self.loss(x_train, y_train)
            self.gradient_descent(x_train, y_train)
            losses.append(loss)
            
            if i % 100 == 0:
                print(f'Iteration: {i}, Loss: {loss}')
        
        fig = px.line(y=losses, title="Loss vs Iteration", template="plotly_dark")
        fig.update_layout(
            title_font_color="#41BEE9",
            xaxis=dict(color="#41BEE9", title="Iterations"),
            yaxis=dict(color="#41BEE9", title="Loss")
        )

        fig.show()
            
data = pd.read_csv('../data/Salary_dataset.csv')
test_data = pd.DataFrame({
    'YearsExperience': [1.0, 1.8, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.0, 10.0, 11.0, 12.0],
    'Salary':          [41000, 38500, 55000, 52000, 71000, 69000, 79000, 102000, 95000, 118000, 108000, 125000, 142000]
})

x_train = data['YearsExperience']
y_train = data['Salary']
x_test = test_data['YearsExperience']
y_test = test_data['Salary']

fig = px.scatter(data, x='YearsExperience', y='Salary', title="Salary vs Years Experience")
fig.show() #Plot out the points of the data

x_train_scaled, x_test_scaled = standardize(x_train, x_test)
y_train_scaled, y_test_scaled = standardize(y_train, y_test)


model = LinearRegression(.01)
model.fit(x_train_scaled, y_train_scaled, 500)

y_pred_scaled = model.predict(x_train_scaled)
y_pred = (y_pred_scaled * np.std(y_train)) + np.mean(y_train)
fig.add_scatter(x=data['YearsExperience'], y=y_pred, mode='lines', name='fit')
fig.show()
        

Iteration: 0, Loss: 0.5030152786789912
Iteration: 100, Loss: 0.08603202534281668
Iteration: 200, Loss: 0.030164744639620853
Iteration: 300, Loss: 0.022679664535808188
Iteration: 400, Loss: 0.021676815937213566


In [49]:
#Testing model using test data

fig = px.scatter(test_data, x='YearsExperience', y='Salary', title='Test Data')
fig.show() #Plot out the points of the data

y_pred_scaled = model.predict(x_test_scaled)
y_pred = (y_pred_scaled * np.std(y_train)) + np.mean(y_train)

fig.add_scatter(x=test_data['YearsExperience'], y=y_pred, mode='lines', name='fit')
fig.show()

